In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import zipfile
import shutil

REPO_DIR = "homework-4-darinkeng"

# Path to the ZIP in your Drive — adjust if you put it elsewhere
REPO_PATH = "/content/drive/MyDrive/DATA342-HW4/homework-4-darinkeng-main"

In [4]:
#Install necessary packages
%pip install -q -r "{REPO_PATH}/requirements.txt"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.8 MB/s eta 0:00:00


In [5]:
import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

from embedding_utils import get_embedding, get_embeddings_batch
from rag_generate import generate_answer

In [6]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import faiss
from tqdm.auto import tqdm

# Paths
RAW_JSON_PATH    = os.path.join(REPO_PATH, "arxiv_clean_100k.json")
CLEAN_PARQUET    = "arxiv_clean.parquet"
CHUNKS_OVERLAP   = "chunks_overlapping.parquet"
CHUNKS_SEMANTIC  = "chunks_semantic.parquet"
EMB_OVERLAP      = "embeddings_overlapping.parquet"
EMB_SEMANTIC     = "embeddings_semantic.parquet"

# Hyperparameters
EMBED_DIM        = 384       # all-MiniLM-L6-v2
BATCH_SIZE       = 256       # embedding batch size
CHUNK_SIZE       = 500       # chars (overlapping)
CHUNK_OVERLAP    = 100       # chars (overlapping)
SEMANTIC_THRESH  = 0.7       # cosine sim threshold for semantic split

## 1. Data Ingestion

In [7]:
def ingest_arxiv(path: str) -> pd.DataFrame:
    """Stream-read the arXiv JSON (JSONL). Skip malformed rows."""
    records = []
    n_bad = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Ingesting JSON"):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                n_bad += 1
    print(f"Loaded {len(records):,} records  |  Skipped {n_bad} malformed rows")
    return pd.DataFrame.from_records(records)

df_raw = ingest_arxiv(RAW_JSON_PATH)
print("Raw shape:", df_raw.shape)

Ingesting JSON: 0it [00:00, ?it/s]

Loaded 100,000 records  |  Skipped 0 malformed rows
Raw shape: (100000, 6)


In [8]:
df_raw

,id,title,abstract,categories,update_date,year
0,0704.0001,Calculation of prompt diphoton production cros...,A fully differential calculation in perturbati...,hep-ph,2008-11-26,2008
1,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-pe...",math.CO cs.CG,2008-12-13,2008
2,0704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is describe...,physics.gen-ph,2008-01-13,2008
3,0704.0004,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle n...,math.CO,2007-05-23,2007
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\Lam...,math.CA math.FA,2013-10-15,2013
...,...,...,...,...,...,...
99995,0812.3926,Nuclear Physics from QCD,Effective field theories provide a bridge betw...,hep-ph,2009-06-25,2009
99996,0812.3927,Mobility Extraction and Quantum Capacitance Im...,The field-effect mobility of graphene devices ...,cond-mat.mes-hall cond-mat.mtrl-sci,2016-11-17,2016
99997,0812.3928,Re-analysis of the First Fringe with 2-Beam in...,We report results from re-analysis of the visi...,astro-ph,2008-12-23,2008
99998,0812.3929,An amplitude-phase (Ermakov-Lewis) approach fo...,In the context of bilayer graphene we use the ...,cond-mat.mtrl-sci math-ph math.MP quant-ph,2009-01-06,2009


## 2. Schema Selection & Cleaning

The dataset comes pre-cleaned with 6 fields. We keep all of them, normalize whitespace, and drop any rows with empty IDs

In [9]:
import re
import numpy as np

KEEP_COLS = ["id", "title", "categories", "abstract", "update_date", "year"]

def normalize_text(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s)
    s = s.replace("\n", " ").replace("\r", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def clean_schema(df):
    df = df.copy()
    for c in KEEP_COLS:
        if c not in df.columns:
            df[c] = ""
    df = df[KEEP_COLS]
    # Normalize text columns (skip 'year' which is numeric)
    for c in ["id", "title", "categories", "abstract", "update_date"]:
        df[c] = df[c].map(normalize_text)
    before = len(df)
    df = df[(df["abstract"].str.len() > 20) & (df["id"] != "")]
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)
    print(f"Cleaning: {before:,} → {len(df):,} rows  (dropped empty abstracts + dupes)")
    return df

df_clean = clean_schema(df_raw)
df_clean.dtypes

Cleaning: 100,000 → 100,000 rows  (dropped empty abstracts + dupes)


,0
id,object
title,object
categories,object
abstract,object
update_date,object
year,int64


## 3. Parquet Storage

In [10]:
import os
import pandas as pd

CLEAN_PARQUET = "arxiv_clean.parquet"
df_clean.to_parquet(CLEAN_PARQUET, engine="pyarrow", compression="snappy", index=False)
print(f"Wrote {CLEAN_PARQUET}  ({os.path.getsize(CLEAN_PARQUET)/1e6:.1f} MB)")

Wrote arxiv_clean.parquet  (51.9 MB)


In [11]:
df_clean = pd.read_parquet(CLEAN_PARQUET)
print("Reloaded shape:", df_clean.shape)

Reloaded shape: (100000, 6)


## 4. Document Chunking — Two Strategies

**(A) Overlapping fixed-size** (500 chars, 100 overlap) and **(B) Semantic chunking** (cosine similarity threshold 0.7 between consecutive sentences).

In [12]:
from embedding_utils import _get_model  # reuse the same encoder

CHUNK_SIZE       = 500
CHUNK_OVERLAP    = 100
SEMANTIC_THRESH  = 0.7

# ---------- Strategy A: Overlapping fixed-size ----------
def overlapping_chunks(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if not text:
        return []
    step = max(1, size - overlap)
    return [text[i:i+size] for i in range(0, len(text), step) if text[i:i+size].strip()]

# ---------- Strategy B: Semantic chunking ----------
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

def semantic_chunks(text, threshold=SEMANTIC_THRESH):
    if not text:
        return []
    sentences = [s.strip() for s in _SENT_SPLIT.split(text) if s.strip()]
    if len(sentences) <= 1:
        return sentences
    model = _get_model()
    embs = model.encode(sentences, normalize_embeddings=True, show_progress_bar=False)
    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        sim = float(np.dot(embs[i-1], embs[i]))   # already L2-normalized → dot == cosine
        if sim < threshold:
            chunks.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])
    chunks.append(" ".join(current))
    return chunks

In [13]:
from tqdm.auto import tqdm

CHUNKS_OVERLAP   = "chunks_overlapping.parquet"
CHUNKS_SEMANTIC  = "chunks_semantic.parquet"

def build_chunk_table(df, chunker, method_name):
    rows = []
    for r in tqdm(df.itertuples(index=False), total=len(df), desc=f"Chunking [{method_name}]"):
        pieces = chunker(r.abstract)
        for j, piece in enumerate(pieces):
            rows.append({
                "chunk_id":    f"{r.id}::{method_name}::{j}",
                "paper_id":    r.id,
                "title":       r.title,
                "categories":  r.categories,
                "chunk_index": j,
                "chunk_text":  piece,
            })
    out = pd.DataFrame(rows)
    print(f"  → {len(out):,} chunks from {len(df):,} papers")
    return out

df_chunks_overlap  = build_chunk_table(df_clean, overlapping_chunks, "overlap")
df_chunks_semantic = build_chunk_table(df_clean, semantic_chunks,    "semantic")

df_chunks_overlap.to_parquet(CHUNKS_OVERLAP, index=False)
df_chunks_semantic.to_parquet(CHUNKS_SEMANTIC, index=False)

print("\nSample overlapping chunk:\n", df_chunks_overlap.iloc[0]["chunk_text"][:300])
print("\nSample semantic chunk:\n",   df_chunks_semantic.iloc[0]["chunk_text"][:300])

Chunking [overlap]:   0%|          | 0/100000 [00:00<?, ?it/s]

  → 249,063 chunks from 100,000 papers


Chunking [semantic]:   0%|          | 0/100000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  → 503,156 chunks from 100,000 papers

Sample overlapping chunk:
 A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs at hadron colliders. All next-to-leading order perturbative contributions from quark-antiquark, gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as all

Sample semantic chunk:
 A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs at hadron colliders.


## 5. Embedding Generation (batched)

In [14]:
EMBED_DIM  = 384       # all-MiniLM-L6-v2
BATCH_SIZE = 256

def embed_chunks(df_chunks, batch_size=BATCH_SIZE):
    texts = df_chunks["chunk_text"].tolist()
    n = len(texts)
    out = np.empty((n, EMBED_DIM), dtype="float32")
    for start in tqdm(range(0, n, batch_size), desc="Embedding"):
        batch = texts[start:start+batch_size]
        vecs = get_embeddings_batch(batch)
        out[start:start+len(batch)] = np.asarray(vecs, dtype="float32")
    return out

print("Embedding overlapping chunks...")
emb_overlap = embed_chunks(df_chunks_overlap)

print("Embedding semantic chunks...")
emb_semantic = embed_chunks(df_chunks_semantic)

print("Overlap embeddings  :", emb_overlap.shape,  emb_overlap.dtype)
print("Semantic embeddings :", emb_semantic.shape, emb_semantic.dtype)

Embedding overlapping chunks...


Embedding:   0%|          | 0/973 [00:00<?, ?it/s]

Embedding semantic chunks...


Embedding:   0%|          | 0/1966 [00:00<?, ?it/s]

Overlap embeddings  : (249063, 384) float32
Semantic embeddings : (503156, 384) float32


## 6. Embedding Storage (Parquet)

In [15]:
EMB_OVERLAP   = "embeddings_overlapping.parquet"
EMB_SEMANTIC  = "embeddings_semantic.parquet"

def save_embeddings(df_chunks, embs, path):
    df_out = df_chunks.copy()
    df_out["embedding"] = list(embs)
    df_out.to_parquet(path, index=False)
    print(f"Wrote {path}  ({os.path.getsize(path)/1e6:.1f} MB)")

save_embeddings(df_chunks_overlap,  emb_overlap,  EMB_OVERLAP)
save_embeddings(df_chunks_semantic, emb_semantic, EMB_SEMANTIC)

Wrote embeddings_overlapping.parquet  (440.4 MB)
Wrote embeddings_semantic.parquet  (832.7 MB)


## 7. FAISS Index Construction

`IndexFlatIP` on L2-normalized vectors = exact cosine similarity.

In [16]:
import faiss

def build_faiss(embs):
    faiss.normalize_L2(embs)  # defensive
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index

index_overlap  = build_faiss(emb_overlap)
index_semantic = build_faiss(emb_semantic)

print("FAISS overlap index size :", index_overlap.ntotal)
print("FAISS semantic index size:", index_semantic.ntotal)

FAISS overlap index size : 249063
FAISS semantic index size: 503156


## Pipeline verification evidence

In [17]:
print("="*60)
print("PIPELINE VERIFICATION")
print("="*60)
print(f"Raw papers ingested        : {len(df_raw):,}")
print(f"Clean papers (post-schema) : {len(df_clean):,}")
print(f"Overlapping chunks         : {len(df_chunks_overlap):,}")
print(f"Semantic chunks            : {len(df_chunks_semantic):,}")
print()
print(f"Overlap embedding shape    : {emb_overlap.shape}  (dtype={emb_overlap.dtype})")
print(f"Semantic embedding shape   : {emb_semantic.shape}  (dtype={emb_semantic.dtype})")
print(f"Embedding dim consistent?  : {emb_overlap.shape[1] == emb_semantic.shape[1] == EMBED_DIM}")
print()
print(f"FAISS overlap.ntotal       : {index_overlap.ntotal}   (matches chunks? {index_overlap.ntotal == len(df_chunks_overlap)})")
print(f"FAISS semantic.ntotal      : {index_semantic.ntotal}   (matches chunks? {index_semantic.ntotal == len(df_chunks_semantic)})")
print()
sample_norm = float(np.linalg.norm(emb_overlap[0]))
print(f"Sample vector L2 norm      : {sample_norm:.6f}  (should be ~1.0)")

PIPELINE VERIFICATION
Raw papers ingested        : 100,000
Clean papers (post-schema) : 100,000
Overlapping chunks         : 249,063
Semantic chunks            : 503,156

Overlap embedding shape    : (249063, 384)  (dtype=float32)
Semantic embedding shape   : (503156, 384)  (dtype=float32)
Embedding dim consistent?  : True

FAISS overlap.ntotal       : 249063   (matches chunks? True)
FAISS semantic.ntotal      : 503156   (matches chunks? True)

Sample vector L2 norm      : 1.000000  (should be ~1.0)


## 8. & 9. Query Embedding + Top-k Retrieval




In [18]:
def retrieve(query, index, df_chunks, k):
    """Embed the query, search FAISS, return top-k chunks with metadata + score."""
    q_vec = np.asarray(get_embedding(query), dtype="float32").reshape(1, -1)
    faiss.normalize_L2(q_vec)
    scores, idxs = index.search(q_vec, k)
    scores, idxs = scores[0], idxs[0]
    rows = []
    for rank, (i, s) in enumerate(zip(idxs, scores), start=1):
        if i < 0:
            continue
        r = df_chunks.iloc[int(i)]
        rows.append({
            "query":      query,
            "rank":       rank,
            "chunk_id":   r["chunk_id"],
            "paper_id":   r["paper_id"],
            "title":      r["title"],
            "score":      float(s),
            "chunk_text": r["chunk_text"],
        })
    return pd.DataFrame(rows)

## Pipeline verification evidence (Show 3 examples)

In [23]:


print("="*80)
print("SAMPLE 1: Overlapping strategy, on-topic physics query, k=3")
print("="*80)
sample1 = retrieve("How is entanglement entropy computed for anti-de Sitter black holes?",
                   index_overlap, df_chunks_overlap, k=3)
display(sample1[["rank", "score", "paper_id", "title", "chunk_text"]])

print("\n" + "="*80)
print("SAMPLE 2: Semantic strategy, same query, k=3 (compare against Sample 1)")
print("="*80)
sample2 = retrieve("How is entanglement entropy computed for anti-de Sitter black holes?",
                   index_semantic, df_chunks_semantic, k=3)
display(sample2[["rank", "score", "paper_id", "title", "chunk_text"]])

print("\n" + "="*80)
print("SAMPLE 3: Semantic strategy, off-topic query, k=3 (low scores expected)")
print("="*80)
sample3 = retrieve("What is the capital of France?",
                   index_semantic, df_chunks_semantic, k=3)
display(sample3[["rank", "score", "paper_id", "title", "chunk_text"]])

SAMPLE 1: Overlapping strategy, on-topic physics query, k=3


,rank,score,paper_id,title,chunk_text
0,1,0.833425,0709.0163,Induced gravity and entanglement entropy of 2D black holes,"Using the fact that 2D Newton constant is wholly induced by a conformal field theory, we derive a formula for the entanglement entropy o..."
1,2,0.795885,0704.0140,Entanglement entropy of two-dimensional Anti-de Sitter black holes,Using the AdS/CFT correspondence we derive a formula for the entanglement entropy of the anti-de Sitter black hole in two spacetime dime...
2,3,0.729662,0804.2724,Entropy function and higher derivative corrections to entropies in (anti-)de Sitter space,We first briefly discuss the relation between black hole thermodynamics and the entropy function formalism. We find that an equation whi...



SAMPLE 2: Semantic strategy, same query, k=3 (compare against Sample 1)


,rank,score,paper_id,title,chunk_text
0,1,0.840743,0704.0140,Entanglement entropy of two-dimensional Anti-de Sitter black holes,Using the AdS/CFT correspondence we derive a formula for the entanglement entropy of the anti-de Sitter black hole in two spacetime dime...
1,2,0.831700,0709.0163,Induced gravity and entanglement entropy of 2D black holes,"Using the fact that 2D Newton constant is wholly induced by a conformal field theory, we derive a formula for the entanglement entropy o..."
2,3,0.773498,0705.2892,Non-supersymmetric Attractor with the Cosmological Constant,Based on the simple observation that the near-horizon geometry of a generic extremal black hole contains two-dimensional anti-de Sitter ...



SAMPLE 3: Semantic strategy, off-topic query, k=3 (low scores expected)


,rank,score,paper_id,title,chunk_text
0,1,0.611375,0712.1456,Detecting abrupt changes of the long-range dependence or the self-similarity of a Gaussian process,"Paris, Ser."
1,2,0.611375,0704.1487,"Wavelet frames, Bergman spaces and Fourier transforms of Laguerre functions","Paris, Ser."
2,3,0.595072,0704.0537,Linearisation of finite abelian subgroups of the Cremona group of the plane,"Paris, S\'er."


##  Evaluation Queries — both strategies × k = {1, 5, 20}

In [20]:
EVAL_QUERIES = [
    "How is entanglement entropy computed for anti-de Sitter black holes?",
    "How are quasinormal frequencies of Schwarzschild black holes calculated using monodromy methods?",
    "What is the capital of France?",
    "What are the health benefits of drinking green tea daily?",
]
K_VALUES = [1, 5, 20]

STRATEGIES = {
    "overlapping": (index_overlap,  df_chunks_overlap),
    "semantic":    (index_semantic, df_chunks_semantic),
}

all_results = []
for strat_name, (idx, dfc) in STRATEGIES.items():
    for q in EVAL_QUERIES:
        for k in K_VALUES:
            res = retrieve(q, idx, dfc, k=k)
            res["strategy"] = strat_name
            res["k"]        = k
            all_results.append(res)

results_df = pd.concat(all_results, ignore_index=True)
results_df = results_df[["strategy", "k", "query", "rank", "chunk_id",
                         "paper_id", "title", "score", "chunk_text"]]
results_df.to_csv("retrieval_results.csv", index=False)
print("Total result rows:", len(results_df))
results_df.head()

Total result rows: 208


,strategy,k,query,rank,chunk_id,paper_id,title,score,chunk_text
0,overlapping,1,How is entanglement entropy computed for anti-...,1,0709.0163::overlap::0,0709.0163,Induced gravity and entanglement entropy of 2D...,0.833425,Using the fact that 2D Newton constant is whol...
1,overlapping,5,How is entanglement entropy computed for anti-...,1,0709.0163::overlap::0,0709.0163,Induced gravity and entanglement entropy of 2D...,0.833425,Using the fact that 2D Newton constant is whol...
2,overlapping,5,How is entanglement entropy computed for anti-...,2,0704.0140::overlap::0,0704.0140,Entanglement entropy of two-dimensional Anti-d...,0.795885,Using the AdS/CFT correspondence we derive a f...
3,overlapping,5,How is entanglement entropy computed for anti-...,3,0804.2724::overlap::0,0804.2724,Entropy function and higher derivative correct...,0.729662,We first briefly discuss the relation between ...
4,overlapping,5,How is entanglement entropy computed for anti-...,4,0710.2956::overlap::1,0710.2956,Near Extremal Black Hole Entropy as Entangleme...,0.721112,"temperature. From this result, we clarify the ..."


In [21]:
pd.set_option("display.max_colwidth", 140)

for strat in STRATEGIES:
    print("#" * 80)
    print(f"### STRATEGY: {strat}")
    print("#" * 80)
    for q in EVAL_QUERIES:
        for k in K_VALUES:
            block = results_df[(results_df.strategy == strat) &
                               (results_df["query"] == q) &
                               (results_df.k        == k)]
            print(f"\n--- Query: {q!r}  |  k={k} ---")
            display(block[["rank", "score", "paper_id", "title", "chunk_text"]])

################################################################################
### STRATEGY: overlapping
################################################################################

--- Query: 'How is entanglement entropy computed for anti-de Sitter black holes?'  |  k=1 ---


,rank,score,paper_id,title,chunk_text
0,1,0.833425,0709.0163,Induced gravity and entanglement entropy of 2D black holes,"Using the fact that 2D Newton constant is wholly induced by a conformal field theory, we derive a formula for the entanglement entropy o..."



--- Query: 'How is entanglement entropy computed for anti-de Sitter black holes?'  |  k=5 ---


,rank,score,paper_id,title,chunk_text
1,1,0.833425,0709.0163,Induced gravity and entanglement entropy of 2D black holes,"Using the fact that 2D Newton constant is wholly induced by a conformal field theory, we derive a formula for the entanglement entropy o..."
2,2,0.795885,0704.0140,Entanglement entropy of two-dimensional Anti-de Sitter black holes,Using the AdS/CFT correspondence we derive a formula for the entanglement entropy of the anti-de Sitter black hole in two spacetime dime...
3,3,0.729662,0804.2724,Entropy function and higher derivative corrections to entropies in (anti-)de Sitter space,We first briefly discuss the relation between black hole thermodynamics and the entropy function formalism. We find that an equation whi...
4,4,0.721112,0710.2956,Near Extremal Black Hole Entropy as Entanglement Entropy via AdS2/CFT1,"temperature. From this result, we clarify the relation between the thermal entropy and entanglement entropy, which is essential for the ..."
5,5,0.706789,0705.2892,Non-supersymmetric Attractor with the Cosmological Constant,"As a test for the non-supersymmetric attractor mechanism, we consider extremal Reissner-Nordstr\""{o}m-(anti-)de Sitter black holes. Base..."



--- Query: 'How is entanglement entropy computed for anti-de Sitter black holes?'  |  k=20 ---


...

,rank,score,paper_id,title,chunk_text
182,1,0.439613,0801.4820,"Cheap Artificial AB-Mountains, Extraction of Water and Energy from Atmosphere and Change of Regional Climate",Additional benefits: The potential of tapping large amounts of fresh water and energy.



--- Query: 'What are the health benefits of drinking green tea daily?'  |  k=5 ---


,rank,score,paper_id,title,chunk_text
183,1,0.439613,0801.4820,"Cheap Artificial AB-Mountains, Extraction of Water and Energy from Atmosphere and Change of Regional Climate",Additional benefits: The potential of tapping large amounts of fresh water and energy.
184,2,0.438889,0808.1371,Iron Behaving Badly: Inappropriate Iron Chelation as a Major Contributor to the Aetiology of Vascular and Other Progressive Inflammatory...,"This explains, for instance, the decidedly mixed effects of antioxidants that have been observed, etc..."
185,3,0.425491,0801.3539,On the Effects of Idiotypic Interactions for Recommendation Communities in Artificial Immune Systems,"This paper reports the results of work in progress, where we undertake some investigations into the nature of this beneficial effect."
186,4,0.413078,0706.1454,Heterogeneity and Increasing Returns May Drive Socio-Economic Transitions,"For example, as we consider here, some products might carry environmental or `green' benefits."
187,5,0.367715,0709.0592,Transport through a multiply connected interacting meso-system using the Keldysh formalism,The main technical challenge is to deal with the lesser Green function.



--- Query: 'What are the health benefits of drinking green tea daily?'  |  k=20 ---


,rank,score,paper_id,title,chunk_text
188,1,0.439613,0801.4820,"Cheap Artificial AB-Mountains, Extraction of Water and Energy from Atmosphere and Change of Regional Climate",Additional benefits: The potential of tapping large amounts of fresh water and energy.
189,2,0.438889,0808.1371,Iron Behaving Badly: Inappropriate Iron Chelation as a Major Contributor to the Aetiology of Vascular and Other Progressive Inflammatory...,"This explains, for instance, the decidedly mixed effects of antioxidants that have been observed, etc..."
190,3,0.425491,0801.3539,On the Effects of Idiotypic Interactions for Recommendation Communities in Artificial Immune Systems,"This paper reports the results of work in progress, where we undertake some investigations into the nature of this beneficial effect."
191,4,0.413078,0706.1454,Heterogeneity and Increasing Returns May Drive Socio-Economic Transitions,"For example, as we consider here, some products might carry environmental or `green' benefits."
192,5,0.367715,0709.0592,Transport through a multiply connected interacting meso-system using the Keldysh formalism,The main technical challenge is to deal with the lesser Green function.
193,6,0.355512,0810.5755,Non-Invasive Glucose Monitoring Techniques: A review and current trends,Intensive therapy and frequent glucose testing has numerous benefits.
194,7,0.351073,0707.1210,Increased peripheral lipid clearance in an animal model of amyotrophic lateral sclerosis,Counterbalancing for this increase with a high-fat diet extends lifespan and prevents motor neuron loss.
195,8,0.340394,0804.4306,Efficient wave function matching approach for quantum transport calculations,In terms of efficiency it is comparable with the widely used Green's function approach.
196,9,0.339295,0711.0491,Community Detection in Complex Networks Using Genetic Algorithms,The results are promising and in some cases better than previously reported studies.
197,10,0.336153,0810.1204,Nonzero angular momentum pairing correlation in shell model,"When compared with the experimental data, the results are found to be encouraging."


## 10. Answer Generation

In [22]:
answers = []
for strat_name, (idx, dfc) in STRATEGIES.items():
    for q in EVAL_QUERIES:
        for k in K_VALUES:
            res = retrieve(q, idx, dfc, k=k)
            chunks = res["chunk_text"].tolist()
            ans = generate_answer(q, chunks, k=k)
            answers.append({"strategy": strat_name, "k": k, "query": q, "answer": ans})
            print(f"\n[{strat_name} | k={k}] Q: {q}\nA: {ans}\n" + "-"*60)

answers_df = pd.DataFrame(answers)
answers_df.to_csv("generated_answers.csv", index=False)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


[overlapping | k=1] Q: How is entanglement entropy computed for anti-de Sitter black holes?
A: We derive a formula for the entanglement entropy of the anti-de Sitter black hole in two spacetime dimensions by utilizing the fact that 2D Newton constant is wholly induced by a conformal field theory.
------------------------------------------------------------

[overlapping | k=5] Q: How is entanglement entropy computed for anti-de Sitter black holes?
A: The AdS/CFT correspondence is a conformal field theory. The leading term in the large black hole mass expansion of our formula reproduces exactly the Bekenstein-Hawking entropy S_BH, whereas the subleading term behaves as ln S_BH. This subleading term has the universal form typical for the entanglement entropy of physical systems described by effective conformal fields theories (e.g. one-dimensional stati).
------------------------------------------------------------

[overlapping | k=20] Q: How is entanglement entropy computed for anti-d